# GrainTrace - Results

# 1. Purpose

This notebook is dedicated to showcase our models results

# 2. Imports

In [ ]:
# ========================================
# Standard Library
# ========================================
import os
import random
import zipfile
from collections import defaultdict
from pathlib import Path

# ========================================
# Data Manipulation & Numerical Computing
# ========================================
import numpy as np
import pandas as pd
from collections import Counter
# ========================================
# Visualization
# ========================================
import matplotlib.pyplot as plt
import plotly.express as px

# ========================================
# Image Processing
# ========================================
from PIL import Image

# ========================================
# Progress Tracking
# ========================================
from tqdm.auto import tqdm

# ========================================
# PyTorch Core
# ========================================
import torch
import torch.nn as nn
import torch.nn.functional as F

# ========================================
# PyTorch Data Utilities
# ========================================
from torch.utils.data import (
    Dataset,
    DataLoader,
    Sampler,
    BatchSampler,
)

# ========================================
# TorchVision
# ========================================
from torchvision import transforms, models
from torchvision.models import swin_t, Swin_T_Weights

# ========================================
# Scikit-learn
# ========================================
from sklearn.manifold import TSNE
from sklearn.metrics import (
    silhouette_score,
    pairwise_distances,
)
from sklearn.model_selection import train_test_split

# ========================================
# Dimensionality Reduction
# ========================================
import umap.umap_ as umap

# 2. Preprocessing

In [ ]:
transform_resize = transforms.Resize((224 * 3, 224 * 3))
transform_normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

transform_1 = transforms.Compose([
    transform_resize,
    transforms.ToTensor(),
    transform_normalize
])

In [ ]:
CONFIG={"BATCH_SIZE": 18,
        "DEVICE": "cuda" if torch.cuda.is_available() else "cpu"}

# 3. Dataset Loading

In [ ]:
def datasetcreationhelper(dataset_path):
    grouped_images = defaultdict(list)
    for bow_folder in os.listdir(dataset_path):
        bow_path = os.path.join(dataset_path, bow_folder) # Each Bow Folder
        if not os.path.isdir(bow_path):
            continue

        for image_name in os.listdir(bow_path):
            # STRICT FILTER: Only load images with 'RSU' in the name


            image_path = os.path.join(bow_path, image_name)
            grouped_images[bow_folder].append(image_path)

    grouped_images = {k: v for k, v in grouped_images.items() if len(v) > 0}
    return grouped_images # returns paths of grouped images

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 3.1 Train Data

In [ ]:
zip_path = "./drive/MyDrive/cropped_bows_new_objdet.zip"


extract_folder = "/content/mybows"


os.makedirs(extract_folder, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

dataset_path = "/content/my_finetuning"

print("Files extracted successfully!")

Files extracted successfully!


### 3.2 Test Data

In [ ]:
zip_path = "./drive/MyDrive/TESTIMAGES_TRAINEXCLUDED.zip"


extract_folder = "/content/TestImages"


os.makedirs(extract_folder, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

dataset_path = "/content/my_finetuning"

print("Files extracted successfully!")


Files extracted successfully!


### 3.3 Paths Configuration

In [ ]:
train_kaggle_path = '/content/mybows/cropped_bows_new_objdet/cropped_bows'
groupedimages = datasetcreationhelper(train_kaggle_path)

In [ ]:
test_kaggle_path = '/content/TestImages/TESTIMAGES_TRAINEXCLUDED/mybows/TestImages'
test_grouped_images = datasetcreationhelper(test_kaggle_path)

# 4. Models

## 4.1 ResNet50

In [ ]:
class ConResNet50(nn.Module):
    def __init__(self, projection_dim=256, freeze_backbone=True):
        super().__init__()

        self.encoder = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        # Remove classification head
        feature_dim = self.encoder.fc.in_features  # 2048
        self.encoder.fc = nn.Identity()

        if freeze_backbone:
            # freeze everything first
            for param in self.encoder.parameters():
                param.requires_grad = False

            # unfreeze layer3
            for param in self.encoder.layer3.parameters():
                param.requires_grad = True

            # unfreeze layer4
            for param in self.encoder.layer4.parameters():
                param.requires_grad = True


        self.projector = nn.Sequential(
            nn.Linear(feature_dim, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(1024, projection_dim, bias=False))

    def forward(self, x):
        # Backbone feature extraction
        x = self.encoder(x)          # [B, 2048]

        # Projection head
        x = self.projector(x)        # [B, projection_dim]

        # Normalize for contrastive learning
        x = F.normalize(x, dim=1)

        return x

### 4.2 Dinov2 Model

In [ ]:
class DINOv2EmbeddingModel(nn.Module):
    def __init__(
        self,
        embedding_dim=256,
        model_name="dinov2_vits14",
        pretrained=True,
        unfreeze_last_backbone_layer=True
    ):
        super().__init__()

        # Load DINOv2 backbone from torch hub
        # Options:
        # dinov2_vits14  -> small, lighter
        # dinov2_vitb14  -> bigger
        # dinov2_vitl14  -> large, heavy
        # dinov2_vitg14  -> giant, very heavy
        self.backbone = torch.hub.load(
            "facebookresearch/dinov2",
            model_name,
            pretrained=pretrained
        )

        # DINOv2 embedding dimension depends on model size
        if model_name == "dinov2_vits14":
            in_features = 384
        elif model_name == "dinov2_vitb14":
            in_features = 768
        elif model_name == "dinov2_vitl14":
            in_features = 1024
        elif model_name == "dinov2_vitg14":
            in_features = 1536
        else:
            raise ValueError(f"Unknown DINOv2 model name: {model_name}")

        # Freeze the entire backbone first
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Unfreeze only the last transformer block
        if unfreeze_last_backbone_layer:
            for param in self.backbone.blocks[-2].parameters():
                param.requires_grad = True

            # Also unfreeze final norm layer
            for param in self.backbone.norm.parameters():
                param.requires_grad = True

        # Projection head for metric learning
        self.embedding_head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, embedding_dim)
        )

    def forward(self, x):
        """
        x shape: (B, 3, H, W)
        output shape: (B, embedding_dim)
        """

        # DINOv2 returns image-level CLS embedding
        features = self.backbone(x)

        embeddings = self.embedding_head(features)

        # L2-normalized embeddings for triplet/contrastive loss
        embeddings = F.normalize(embeddings, p=2, dim=1)

        return embeddings

# 5. Load Models

## 5.1 Resnet50 Hard Batch

In [ ]:
resnet50_hard = ConResNet50()

checkpoint = torch.load("/content/ResnetTestIncluded.pth")

resnet50_hard.load_state_dict(checkpoint)

resnet50_hard = resnet50_hard.to(CONFIG["DEVICE"])

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 214MB/s]


### 5.2 Dinov2

In [ ]:
Dino_model= DINOv2EmbeddingModel()
checkpoint = torch.load("/content/DinoTestIncluded_moreepochs.pth", map_location=CONFIG["DEVICE"])

Dino_model.load_state_dict(checkpoint)
Dino_model = Dino_model.to(CONFIG["DEVICE"])
Dino_model.eval()

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 266MB/s]


DINOv2EmbeddingModel(
  (backbone): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
      (norm): Identity()
    )
    (blocks): ModuleList(
      (0-11): 12 x NestedTensorBlock(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): MemEffAttention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (drop_path1): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
        (ls2): LayerScale()
        (drop_pa

# 6. Embeddings Generation

In [ ]:
class BowInferenceDataset(Dataset):
    def __init__(self, grouped_images, transform=None):
        self.samples = []
        self.transform = transform

        for label, paths in grouped_images.items():
            for path in paths:
                self.samples.append((path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label, path

In [ ]:
def extract_embeddings_batched(
    grouped_images,
    model,
    transform,
    device,
    batch_size=8,
    num_workers=2
):
    dataset = BowInferenceDataset(grouped_images=grouped_images, transform=transform)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    model.eval()

    all_embeddings = []
    all_labels = []
    all_paths = []

    with torch.no_grad():
        for images, labels, paths in tqdm(loader, desc="Extracting embeddings"):

            images = images.to(device)

            # Forward pass in batch
            embeddings = model(images)

            # Move to CPU numpy
            embeddings = embeddings.detach().cpu().numpy()

            all_embeddings.append(embeddings)
            all_labels.extend(labels)
            all_paths.extend(paths)

    all_embeddings = np.vstack(all_embeddings)

    return all_embeddings, all_labels, all_paths

# 7. Testing

In [ ]:
def save_train_embeddings(
    train_grouped_images,
    model,
    transform,
    device,
    save_path="train_embeddings.pt",
    batch_size=8,
    num_workers=2
):
    train_embeddings, train_labels, train_paths = extract_embeddings_batched(
        grouped_images=train_grouped_images,
        model=model,
        transform=transform,
        device=device,
        batch_size=batch_size,
        num_workers=num_workers
    )

    train_index = {
        "embeddings": train_embeddings,
        "labels": train_labels,
        "paths": train_paths
    }

    torch.save(train_index, save_path)

    print(f"Saved train embeddings to: {save_path}")
    print(f"Number of train images: {len(train_labels)}")

    return train_index

In [ ]:
def predict_nearest_bow_with_cdist(
    test_grouped_images,
    model,
    transform,
    device,
    train_embeddings_path="/content/train_embeddings.pt",
    batch_size=8,
    num_workers=2,
    save_csv_path="/content/test_predictions.csv"
):
    train_index = torch.load(
        train_embeddings_path,
        map_location="cpu",
        weights_only=False
    )

    train_embeddings = torch.tensor(train_index["embeddings"]).float()
    train_labels = train_index["labels"]
    train_paths = train_index["paths"]

    test_embeddings, test_labels, test_paths = extract_embeddings_batched(
        grouped_images=test_grouped_images,
        model=model,
        transform=transform,
        device=device,
        batch_size=batch_size,
        num_workers=num_workers
    )

    test_embeddings = torch.tensor(test_embeddings).float()

    distances = torch.cdist(test_embeddings, train_embeddings, p=2)

    nearest_indices = distances.argmin(dim=1)
    nearest_distances = distances.min(dim=1).values

    results = []

    for i, nearest_idx in enumerate(nearest_indices):
        nearest_idx = nearest_idx.item()

        actual_label = test_labels[i]
        actual_label= "bow_"+ actual_label

        predicted_label = train_labels[nearest_idx]

        results.append({
            "test_path": test_paths[i],
            "actual_label": actual_label,
            "predicted_label": predicted_label,
            "nearest_train_path": train_paths[nearest_idx],
            "euclidean_distance": nearest_distances[i].item(),
            "correct": actual_label == predicted_label
        })

    results_df = pd.DataFrame(results)

    accuracy = results_df["correct"].mean()

    print(f"Total test images: {len(results_df)}")
    print(f"Correct predictions: {results_df['correct'].sum()}")
    print(f"Accuracy: {accuracy:.2%}")

    results_df.to_csv(save_csv_path, index=False)
    print(f"Saved predictions to: {save_csv_path}")

    return results_df, accuracy

In [ ]:
def predict_nearest_bow_with_cdist_top5_mode(
    test_grouped_images,
    model,
    transform,
    device,
    train_embeddings_path="/content/train_embeddings.pt",
    batch_size=8,
    num_workers=2,
    save_csv_path="/content/test_predictions_top5.csv",
    k=5
):
    train_index = torch.load(
        train_embeddings_path,
        map_location="cpu",
        weights_only=False
    )

    train_embeddings = torch.as_tensor(train_index["embeddings"]).float()
    train_labels = train_index["labels"]
    train_paths = train_index["paths"]

    test_embeddings, test_labels, test_paths = extract_embeddings_batched(
        grouped_images=test_grouped_images,
        model=model,
        transform=transform,
        device=device,
        batch_size=batch_size,
        num_workers=num_workers
    )

    test_embeddings = torch.as_tensor(test_embeddings).float()

    # Compute Euclidean distances: shape = [num_test_images, num_train_images]
    distances = torch.cdist(test_embeddings, train_embeddings, p=2)

    # In case training set has fewer than 5 images
    k = min(k, train_embeddings.shape[0])

    # Get nearest k training examples for each test image
    topk_distances, topk_indices = torch.topk(
        distances,
        k=k,
        dim=1,
        largest=False
    )

    results = []

    for i in range(len(test_labels)):

        actual_label = test_labels[i]
        actual_label= "bow_"+ actual_label
        nearest_indices = topk_indices[i].tolist()
        nearest_distances = topk_distances[i].tolist()

        nearest_labels = [train_labels[idx] for idx in nearest_indices]
        nearest_paths = [train_paths[idx] for idx in nearest_indices]

        # Mode of the nearest k labels
        label_counts = Counter(nearest_labels)
        max_count = max(label_counts.values())

        candidate_labels = [
            label for label, count in label_counts.items()
            if count == max_count
        ]

        # Tie-breaker:
        # If two labels appear the same number of times,
        # choose the one whose closest example has the smallest distance.
        if len(candidate_labels) == 1:
            predicted_label = candidate_labels[0]
        else:
            best_label = None
            best_distance = float("inf")

            for label in candidate_labels:
                label_distances = [
                    dist for lbl, dist in zip(nearest_labels, nearest_distances)
                    if lbl == label
                ]

                closest_distance_for_label = min(label_distances)

                if closest_distance_for_label < best_distance:
                    best_distance = closest_distance_for_label
                    best_label = label

            predicted_label = best_label

        results.append({
            "test_path": test_paths[i],
            "actual_label": actual_label,
            "predicted_label": predicted_label,
            "top5_labels": nearest_labels,
            "top5_distances": nearest_distances,
            "top5_train_paths": nearest_paths,
            "correct": actual_label == predicted_label
        })

    results_df = pd.DataFrame(results)

    accuracy = results_df["correct"].mean()

    print(f"Total test images: {len(results_df)}")
    print(f"Correct predictions: {results_df['correct'].sum()}")
    print(f"Top-{k} mode accuracy: {accuracy:.2%}")

    results_df.to_csv(save_csv_path, index=False)
    print(f"Saved predictions to: {save_csv_path}")

    return results_df, accuracy

### 7.1 Resnet50

### Run Inference

In [ ]:
train_index = save_train_embeddings(
    train_grouped_images=groupedimages,
    model=resnet50_hard,
    transform=transform_1,
    device=CONFIG["DEVICE"],
    save_path="/content/train_embeddings.pt",
    batch_size=8
)

Extracting embeddings:   0%|          | 0/264 [00:00<?, ?it/s]

Saved train embeddings to: /content/train_embeddings.pt
Number of train images: 2105


### Nearest Bow-based Accuracy

In [ ]:
predictions_df, accuracy = predict_nearest_bow_with_cdist(
    test_grouped_images=test_grouped_images,
    model=resnet50_hard,
    transform=transform_1,
    device=CONFIG["DEVICE"],
    train_embeddings_path="/content/train_embeddings.pt",
    batch_size=8,
    save_csv_path="/content/test_predictions.csv"
)

Extracting embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

Total test images: 18
Correct predictions: 7
Accuracy: 38.89%
Saved predictions to: /content/test_predictions.csv


### 5 Nearest Bow-based Accuracy

In [ ]:
predictions_df, accuracy = predict_nearest_bow_with_cdist_top5_mode(
    test_grouped_images=test_grouped_images,
    model=resnet50_hard,
    transform=transform_1,
    device=CONFIG["DEVICE"],
    train_embeddings_path="/content/train_embeddings.pt",
    batch_size=8,
    save_csv_path="/content/test_predictions_top5.csv",
    k=5
)

Extracting embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

Total test images: 18
Correct predictions: 7
Top-5 mode accuracy: 38.89%
Saved predictions to: /content/test_predictions_top5.csv


### 7.2 Dinov2

### Run Inference

In [ ]:
train_index = save_train_embeddings(
    train_grouped_images=groupedimages,
    model=Dino_model,
    transform=transform_1,
    device=CONFIG["DEVICE"],
    save_path="/content/train_embeddings.pt",
    batch_size=8
)

Extracting embeddings:   0%|          | 0/264 [00:00<?, ?it/s]

Saved train embeddings to: /content/train_embeddings.pt
Number of train images: 2105


### Nearest Bow-based Accuracy

In [ ]:
predictions_df, accuracy = predict_nearest_bow_with_cdist(
    test_grouped_images=test_grouped_images,
    model=Dino_model,
    transform=transform_1,
    device=CONFIG["DEVICE"],
    train_embeddings_path="/content/train_embeddings.pt",
    batch_size=8,
    save_csv_path="/content/test_predictions_dino.csv"
)

Extracting embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

Total test images: 18
Correct predictions: 15
Accuracy: 83.33%
Saved predictions to: /content/test_predictions_dino.csv


### 5 Nearest Bow-based Accuracy

In [ ]:
predictions_df, accuracy = predict_nearest_bow_with_cdist_top5_mode(
    test_grouped_images=test_grouped_images,
    model=Dino_model,
    transform=transform_1,
    device=CONFIG["DEVICE"],
    train_embeddings_path="/content/train_embeddings.pt",
    batch_size=8,
    save_csv_path="/content/test_predictions_top5_dino.csv",
    k=5
)

Extracting embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

Total test images: 18
Correct predictions: 15
Top-5 mode accuracy: 83.33%
Saved predictions to: /content/test_predictions_top5_dino.csv
